<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-06-function-calling/lesson-6.4-langchain-tool-loop/notebooks/GCP_Capstone_6.4_LangChain.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 6.4 Rebuild DocuMind's Tool Loop in LangChain
**Netsetos GenAI Engineering - GCP Capstone** | Module 6 | new in v1.1

Lesson 6.2 had you write the tool-calling loop by hand, then build a registry to guard it. This notebook deletes that loop and gets it back from a framework in one line.

The interesting part is not the line. It is the audit of what disappeared with it:

- a Gemini chat model on **Vertex AI** through LangChain, and the argument that decides whether it constructs at all
- tool schemas generated from **docstrings** instead of written twice
- `create_agent` in place of the hand-written loop - and the **dispatch seam it does not have**
- lesson 6.2's registry, put back as **middleware**
- a **Chroma** dev lane and `DOCUMIND_PROFILE=local`, so the agent runs with no cloud credentials
- a **parity test** on tool calls, because a port you cannot diff is a port you cannot trust

Prerequisites: lessons **6.1-6.3** (the same three DocuMind tools, unchanged). *API facts verified 2026-09-04 against langchain 1.4.0, langchain-google-genai 4.4.0.*

## Setup
One switch decides whether this notebook talks to Google Cloud at all. Everything runs on the `gcp` profile until Cell 7 builds the local half.

In [ ]:
!pip install -q "langchain==1.4.0" "langchain-google-genai==4.4.0" \
                "langchain-chroma==1.1.0" "chromadb==1.5.9" "ddgs==9.16.0" \
                "langchain-ollama==1.1.0"

from google.colab import auth
auth.authenticate_user()

import os
PROJECT_ID = "documind-ai-YOUR-ID"      # CHANGE THIS
LOCATION   = "global"                    # Gemini 3.x generation is served ONLY from global
USD_INR    = 85                          # course-wide conversion rate

# One switch decides whether this notebook talks to Google Cloud at all.
#   gcp   - Gemini on Vertex AI + the shared index          (what production runs)
#   local - a local model + a Chroma directory on disk      (what you develop against)
# Step 7 builds the local half. Until then everything runs on gcp.
os.environ.setdefault("DOCUMIND_PROFILE", "gcp")
os.environ.setdefault("GOOGLE_CLOUD_PROJECT", PROJECT_ID)   # Step 7's build_llm() reads this
PROFILE = os.environ["DOCUMIND_PROFILE"]
print("profile:", PROFILE, "| project:", PROJECT_ID)

## Cell 1: The model object
Three arguments carry the whole configuration, and two of them are the difference between working and a stack trace on line one.

> **`project=` is not optional in practice.** Leave it out and the constructor resolves Application Default Credentials *there and then*, so a missing credential raises `DefaultCredentialsError` on the line that **builds** the object - not on the first request, where you would look for it. Reproduced against langchain-google-genai 4.4.0.

`ChatVertexAI` is the deprecated class: since langchain-google-vertexai 3.2.0 it defers to `ChatGoogleGenerativeAI(vertexai=True)`, because both now sit on the same unified `google-genai` SDK.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# ChatGoogleGenerativeAI, NOT ChatVertexAI. Since langchain-google-vertexai 3.2.0 the
# ChatVertexAI class is deprecated in favour of this one with vertexai=True, because both
# now sit on the same unified google-genai SDK underneath - the same consolidation the
# course made in lesson 1.1 when vertexai.generative_models went away.
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    vertexai=True,               # route to Vertex AI, not the Developer API
    project=PROJECT_ID,          # REQUIRED - see the note below
    location=LOCATION,           # "global": Gemini 3.x generation is not served regionally
    thinking_level="low",        # minimal | low | medium | high
)

# Three things worth knowing before you copy this into a service.
#
# 1. project= is not optional in practice. Leave it out and the constructor resolves
#    Application Default Credentials THERE AND THEN - so a missing credential blows up on
#    the line that builds the object, not on the first request. Reproduced on 2026-09-04:
#    DefaultCredentialsError at construction. In Colab, auth.authenticate_user() must run first.
#
# 2. thinking_level replaces thinking_budget. On Gemini 2.5 you spent a token budget; on
#    Gemini 3 you choose a level, and setting both logs a warning and ignores the budget.
#    "low" is the right default for a tool loop: the reasoning that matters is which tool
#    to call, and that is a short decision.
#
# 3. No temperature. Gemini 3.x ignores the sampling parameters (lesson 1.3), so setting one
#    buys you nothing but the belief that you tuned something.

print(llm.model, "| thinking:", llm.thinking_level)

## Cell 2: Tools from docstrings
The same three tools as lessons 6.1-6.3, with the identical signature. That is the point: the tool layer does not change when the framework does.

> The docstring stops being a comment and becomes an interface. The model reads the `Args` block at every call, so a stale docstring is a stale API contract - and it fails the way bad prompts fail, by producing plausible wrong arguments rather than an error.

In [ ]:
USD_INR = 85          # course-wide conversion rate
RATES = {"standard": 0.05, "priority": 0.12, "bulk": 0.03}

# DocuMind's demo corpus - the SAME five documents in every
# lesson of Modules 6, 7 and 8, so results stay comparable.
# 119 pages in total, which is what the cost examples bill.
GS = "gs://documind-acme"
_DOCS = [
    # doc_id, doc_type, page, score, quote
    ("hr_policy_2026", "policy", 12, 0.94,
     "A senior engineer serves a notice period of 60 days."),
    ("hr_policy_2026", "policy", 31, 0.81,
     "Earned leave is encashed on exit, capped at 45 days."),
    ("msa_acme_2026", "contract", 8, 0.88,
     "Either party may terminate on 90 days written notice."),
    ("inv_2026_0412", "invoice", 1, 0.76,
     "Total payable Rs 1,84,500, inclusive of 18% GST."),
    ("gstr1_q1_fy27", "form", 4, 0.68,
     "Outward taxable supplies for the quarter, GSTR-1."),
    ("rag_survey_2026", "research_paper", 6, 0.72,
     "Hybrid retrieval mixes dense and sparse signals."),
]
CORPUS = [{"chunk_id": f"{d}#{p}", "doc_type": t, "page": p,
           "source_uri": f"{GS}/{d}.pdf", "quote": q,
           "score": s} for d, t, p, s, q in _DOCS]
CITATION_FIELDS = ("chunk_id", "source_uri", "page", "quote",
                   "score")

# Document lengths, for the tool-chaining demos: retrieve
# finds chunks, and the cost tool bills whole documents.
DOC_PAGES = {"hr_policy_2026": 48, "msa_acme_2026": 32,
             "inv_2026_0412": 3, "gstr1_q1_fy27": 12,
             "rag_survey_2026": 24}          # 119 pages


def docs_of(citations: list) -> dict:
    """Distinct source documents behind a set of citations."""
    return {c["chunk_id"].split("#")[0]: True
            for c in citations}

from langchain_core.tools import tool

# The same DocuMind tools as lessons 6.1-6.3, and the point of the exercise is that the tool
# layer does not change when the framework does.
#
# Be precise about "the same", because the module is not uniform and pretending otherwise would
# repeat the defect this module opened with. calculate_processing_cost IS identical everywhere
# after the Module 6 fixes. retrieve is NOT: 6.1 and 6.2 take (query, doc_type, top_k),
# 6.3 dropped top_k for its parallel-calls demo. This lesson follows 6.1/6.2.
#
# That drift is worth naming rather than hiding: it is exactly what a schema generated from ONE
# function definition prevents, which is the argument Step 1 made for the framework.
@tool
def retrieve(query: str, doc_type: str = "all",
             top_k: int = 5) -> dict:
    """Retrieve grounded passages from DocuMind's corpus.

    Args:
        query: The question, in natural language
        doc_type: policy, contract, invoice, form,
            research_paper, or all
        top_k: How many passages to return
    """
    # A mock, but not a stub: it really filters and ranks, so
    # a question the corpus cannot answer returns NOTHING and
    # answerable=False. A mock that always succeeds teaches
    # that retrieval always succeeds - the one thing it never
    # does.
    # doc_type is one of the two keys the live API takes in
    # `filters` (doc_type, kind); any other key is a 400, and
    # the tenant is never a filter - the roster sets it (12.2).
    words = {w for w in query.lower().split() if len(w) > 3}
    hits = [c for c in CORPUS
            if doc_type in ("all", c["doc_type"])
            and any(w in c["quote"].lower() for w in words)]
    hits.sort(key=lambda c: -c["score"])
    hits = hits[:top_k]
    top = hits[0]["score"] if hits else 0.0
    return {
        "citations": [{k: c[k] for k in CITATION_FIELDS}
                      for c in hits],
        "answerable": bool(hits),
        "confidence": ("high" if top >= 0.85 else
                       "medium" if hits else "low"),
    }


@tool
def calculate_processing_cost(
        total_pages: int, num_documents: int = 1,
        processing_type: str = "standard") -> dict:
    """Estimate document processing cost in USD and INR.

    Args:
        total_pages: Total page count across all documents
        num_documents: How many documents those pages span
        processing_type: standard, priority, or bulk
    """
    rate = RATES.get(processing_type, RATES["standard"])
    cost = total_pages * rate
    return {"num_documents": num_documents,
            "total_pages": total_pages,
            "processing_type": processing_type,
            "rate_per_page": rate,
            "cost_usd": round(cost, 2),
            "cost_inr": round(cost * USD_INR, 2)}


@tool
def get_usage_stats(metric: str, days: int = 7) -> dict:
    """Get DocuMind pipeline usage statistics.

    Args:
        metric: queries, costs, latency, or users
        days: Number of days to look back
    """
    mock = {"queries": 1247, "costs": 18.50,
            "latency": 245, "users": 42}
    return {"metric": metric, "period": f"last {days} days",
            "value": mock.get(metric, 0), "trend": "+12%"}


@tool
def delete_document(doc_id: str) -> dict:
    """Permanently delete a document from the DocuMind collection.

    Args:
        doc_id: The document identifier, e.g. hr_policy_2026
    """
    # Never reached in this lesson: Step 5's guard refuses it before dispatch.
    raise AssertionError("delete_document executed - the guard did not run")


# delete_document IS bound, on purpose. A model that does not know the capability exists cannot
# tell a user "I can do that, but it needs an approval"; it just says no and looks broken. Binding
# it and gating it are different decisions, and you want the second one, not the first.
#
# The corollary matters more: a blocked list naming tools you never bound is not a control, it is
# a comment. The model could not have called them anyway.
TOOLS = [retrieve, calculate_processing_cost, get_usage_stats, delete_document]

## Cell 3: Read the schema it generated
Compare this with the `FunctionDeclaration` you typed by hand in lesson 6.1. Same name, same parameters, same required list - derived instead of written.

> The win is not elegance. It is that the tool now has exactly **one** definition, so the drift this module opened with - three different shapes of the same tool across three lessons - cannot recur.

In [ ]:
import json

# Read what LangChain generated, and compare it with the FunctionDeclaration you wrote BY HAND
# in lesson 6.1. Same name, same parameter names, same types, same required list - derived from
# the signature and the docstring instead of typed out.
print(json.dumps(calculate_processing_cost.args_schema.model_json_schema(), indent=2))

# This is the actual saving, and it is worth being precise about what it is. You did not get a
# better schema. You got ONE definition of the tool instead of two that drift - which is exactly
# the defect this module started with, where the same tool had three different shapes across
# three lessons.
#
# What it costs: the docstring is now load-bearing. Delete the Args block and the model loses the
# per-parameter descriptions it uses to choose values. A docstring is documentation everywhere
# else in your codebase; here it is a prompt.

## Cell 4: The loop you no longer write
`create_agent` is lesson 6.2's while-loop, plus a message history you did not have to model, checkpointing, streaming and a middleware hook.

In [ ]:
from langchain.agents import create_agent

SYSTEM = ("You are DocuMind AI, a document intelligence assistant. "
          "Use the tools for document questions. When estimating costs, search first "
          "so the page count is real rather than guessed.")

agent = create_agent(model=llm, tools=TOOLS, system_prompt=SYSTEM)

result = agent.invoke({"messages": [
    {"role": "user", "content": "Find all legal documents and estimate bulk processing cost"}]})

for msg in result["messages"]:
    kind = msg.__class__.__name__
    if getattr(msg, "tool_calls", None):
        for tc in msg.tool_calls:
            print(f"  [tool call] {tc['name']}({tc['args']})")
    elif kind == "ToolMessage":
        print(f"  [tool result] {msg.content[:90]}")
    elif msg.content:
        print(f"  [{kind}] {str(msg.content)[:160]}")

In [ ]:
# Lesson 6.2's loop, in full, was roughly this shape:
#
#   for turn in range(max_turns):
#       response = client.models.generate_content(...)
#       contents.append(response.candidates[0].content)
#       if not response.function_calls:
#           return response.text
#       for fc in response.function_calls:
#           payload = dispatch(fc.name, fc.args)          <-- the seam you built
#           result_parts.append(Part.from_function_response(...))
#       contents.append(Content(role="user", parts=result_parts))
#
# create_agent is that loop. It also brings a message history you did not have to model,
# checkpointing, streaming, and a middleware hook. Those are real.
#
# Now find the seam. It is gone. There is no dispatch= parameter on create_agent, because
# the agent calls the tool objects directly - and lesson 6.2 spent a whole step establishing
# that a registry whose execute() you route around is not a control.
#
# So the question this step exists to answer: where does the guard live now?

## Cell 5: Putting the guard back
Lesson 6.2 established that a registry whose `execute()` you route around is not a control, then built a `dispatch` seam so every call had to pass through it.

> **`create_agent` has no `dispatch` parameter.** It calls the tool objects directly. Port the loop as written and you re-introduce the exact bug 6.2 was written to fix.

`wrap_tool_call` sits between the agent's decision to call a tool and the call itself - precisely where `registry.execute()` sat. The framework did not remove the control. It moved where controls can live.

In [ ]:
import json, logging, time

from langchain.agents.middleware import AgentMiddleware
from langchain_core.messages import ToolMessage

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("documind")

BLOCKED = {"delete_document", "send_email", "modify_access"}
TIMEOUTS = {"retrieve": 30, "calculate_processing_cost": 10, "get_usage_stats": 60}


class GuardMiddleware(AgentMiddleware):
    """The FunctionRegistry from 6.2, relocated to where this framework can see it.

    wrap_tool_call sits between the agent's decision to call a tool and the call itself,
    which is precisely where registry.execute() sat. Same three jobs: refuse the blocked
    ones, bound the slow ones, record all of them.
    """

    def wrap_tool_call(self, request, handler):
        name = request.tool_call["name"]
        if name in BLOCKED:
            # RETURN TYPE MATTERS. The hook is annotated -> ToolMessage | Command, and the
            # value goes straight into the message list. Return a bare dict - the obvious
            # thing, and what 6.2's dispatch seam returned - and LangChain raises
            # "Message dict must contain 'role' and 'content' keys" before the model sees
            # anything. Errors are still data, as in 6.2; they just have to be a ToolMessage.
            return ToolMessage(
                content=json.dumps({"error": f"{name} requires manual approval"}),
                name=name, tool_call_id=request.tool_call["id"], status="error")
        started = time.monotonic()
        try:
            return handler(request)
        finally:
            elapsed = time.monotonic() - started
            logger.info("%s took %.2fs (budget %ss)", name, elapsed, TIMEOUTS.get(name, 30))


guarded = create_agent(model=llm, tools=TOOLS, system_prompt=SYSTEM,
                       middleware=[GuardMiddleware()])

# Prove the guard is load-bearing rather than decorative. delete_document is BOUND, so the model
# really does emit the call and the guard really does refuse it - read the ToolMessage, not just
# the final prose, because prose alone cannot tell you whether the tool ran.
out = guarded.invoke({"messages": [
    {"role": "user", "content": "Delete document hr_policy_2026"}]})
for m in out["messages"]:
    if isinstance(m, ToolMessage):
        print(f"  [tool result] status={m.status} {m.content}")
print("  [final]", out["messages"][-1].content[:200])

# If delete_document had NOT been in TOOLS, this cell would print a polite refusal written by the
# model and the guard would never have run. That is the trap: a demo that looks identical whether
# or not the control works. Check the ToolMessage.
#
# The lesson to carry: a framework does not remove the controls you need, it MOVES them.
# Port the loop without porting the guard and you have shipped the 6.2 bug on purpose.

## Cell 6: A vector store you can run on a train
Chroma on local disk, so the retrieval half of the loop works with no cloud.

> Note the two clients again: the chat model is on `global` because Gemini 3.x generation is served only from there, and the embedding model is on `us-central1` because embeddings are regional-only. One client cannot do both.

In [ ]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Embeddings are REGIONAL-ONLY - the rule from lesson 1.1 that Module 5 kept running into.
# The chat model above is on "global"; this one must not be.
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    vertexai=True, project=PROJECT_ID, location="us-central1",
    output_dimensionality=768)   # NOT optional - the model returns 3072 unless you ask

# The same passages the rest of the course retrieves, so a query here and a
# query in 6.1 or 8.5 return the same thing from a different backend.
CHUNKS = [
    ("hr_policy_2026", "A senior engineer serves a notice period of 60 days."),
    ("msa_acme_2026", "Either party may terminate on 90 days written notice."),
    ("inv_2026_0412", "Total payable Rs 1,84,500, inclusive of 18% GST."),
]

store = Chroma(
    collection_name="documind_dev",
    embedding_function=embeddings,
    persist_directory="./documind_chroma",   # a directory, not a service
)
# One chunk per call. gemini-embedding-001 on Vertex accepts a SINGLE text per request
# (lesson 2.2), and Chroma's add_texts batches internally with no way to reach batch_size -
# so a three-text call is one request with three inputs, and it 400s.
for _doc_id, _text in CHUNKS:
    store.add_texts(texts=[_text], metadatas=[{"doc_id": _doc_id}], ids=[_doc_id])

# NOTE: this returns a DISTANCE, not a similarity - lower is closer. Read it as a similarity
# and your ranking is exactly backwards, which is a bug that looks like a bad embedding model.
for doc, distance in store.similarity_search_with_score("how long is the notice period", k=2):
    print(f"  distance={distance:.3f}  {doc.metadata['doc_id']}  {doc.page_content[:60]}")

In [ ]:
# WHY THIS IS A DEVELOPMENT LANE AND NOT A SMALL PRODUCTION ONE
#
# persist_directory is a path on local disk. On Cloud Run that disk is:
#   - per-instance, so two instances hold two different indexes and answers depend on routing
#   - in-memory by default, so it is charged as RAM and vanishes on every deploy and scale-down
#
# People do ship this by accident. It works in staging with one instance, then quietly returns
# different answers per request the moment traffic justifies a second. If you must run it on
# Cloud Run at all, --max-instances 1 makes the failure mode impossible rather than intermittent:
#
#   gcloud run deploy documind-dev --max-instances 1 --region asia-south1
#
# The real store is the Vector Search index from Module 4, with the filters from lesson 5.5.
# Chroma is here so that a laptop with no network can still run the whole loop.

## Cell 7: One switch, two lanes
`DOCUMIND_PROFILE=local` swaps the model and the store. The agent code below the switch does not change - and the day a profile needs its own agent code, the switch has failed and you are maintaining two applications under one name.

> A 4B local model calls the wrong tool more often than Gemini. Treat that as the feature it is: it is a harsher reader of your tool descriptions, and the descriptions are the part you control.

In [ ]:
# documind/profile.py - one switch, two lanes.
#
# The point is NOT that a local model is as good. It is not. The point is that a broken tool
# loop, a bad prompt, or a wrong schema can be found on a train with no signal, and that a new
# engineer can run the thing on their first morning without a billing account.

import os

PROFILE = os.environ.get("DOCUMIND_PROFILE", "gcp")


def build_llm():
    if PROFILE == "local":
        # Ollama, a model small enough for a laptop. Tool calling on a 4B model is noticeably
        # worse than Gemini - it will sometimes call the wrong tool. That is a feature here:
        # the local lane is where you find out your tool DESCRIPTIONS are ambiguous, because a
        # weaker model is a harsher reader of them.
        from langchain_ollama import ChatOllama
        return ChatOllama(model="gemma3:4b", temperature=0)
    from langchain_google_genai import ChatGoogleGenerativeAI
    return ChatGoogleGenerativeAI(model="gemini-3.6-flash", vertexai=True,
                                  project=os.environ["GOOGLE_CLOUD_PROJECT"],
                                  location="global", thinking_level="low")


def build_store(embeddings=None):
    if PROFILE == "local":
        from langchain_chroma import Chroma
        # DeterministicFakeEmbedding, not FakeEmbeddings: the deterministic one returns the
        # same vector for the same text, so a store you write and then query gives stable
        # results. Both live in langchain_core, which is already a dependency - reaching for
        # langchain_community here would pull a package the lesson never installs.
        from langchain_core.embeddings import DeterministicFakeEmbedding
        return Chroma(collection_name="documind_dev",
                      embedding_function=embeddings or DeterministicFakeEmbedding(size=768),
                      persist_directory="./documind_chroma")
    raise NotImplementedError("gcp profile uses the Vector Search index from Module 4")


# The contract that makes this worth having: the AGENT code below this line is identical in
# both lanes. If a profile ever needs its own agent code, the switch has failed and you have
# two applications wearing one name.
agent = create_agent(model=build_llm(), tools=TOOLS, system_prompt=SYSTEM,
                     middleware=[GuardMiddleware()])

## Cell 8: Two ways to reach the web
Gemini's own grounding binds as a plain dict, not a callable. A tool you own is ordinary code.

> Built-in grounding bills **per search query** on Gemini 3 - one prompt that searches four times costs four - and combining built-in tools with your own function declarations is a **Preview** capability, Gemini 3 only. Neither is 'the answer'; the routing is.

In [ ]:
# Two ways to let the agent reach the open web, and they are not interchangeable.

# A. Gemini's OWN grounding, bound as a built-in tool. Google runs the search server-side and
#    returns grounded text plus citations. Built-in tools bind as plain dicts, not callables:
grounded = llm.bind_tools([{"google_search": {}}])
answer = grounded.invoke("What are the 2026 GDPR enforcement trends?")
print(answer.content[:300])

# B. A search tool YOU own, as an ordinary LangChain tool. You choose the provider, you see the
#    query, you can cache and log it, and it runs on the local profile too.
from ddgs import DDGS

@tool
def web_search(query: str, max_results: int = 5) -> list:
    """Search the public web for information not in DocuMind's own documents.

    Args:
        query: What to search for
        max_results: How many results to return
    """
    with DDGS() as ddgs:
        return [{"title": r["title"], "url": r["href"], "snippet": r["body"][:200]}
                for r in ddgs.text(query, max_results=max_results)]


# ROUTE ONE FAMILY PER TURN. Two agents, one per tool family, and a cheap decision about which
# to call. This is not a workaround for the Vertex limitation below - it is the shape that
# survives it, and it is also how you keep a web-search bill from being a surprise.
web_agent = create_agent(model=llm, tools=[web_search], middleware=[GuardMiddleware()],
                         system_prompt="Answer from web search results. Always cite the URL.")

EXTERNAL = ("regulation", "gdpr", "law", "benchmark", "industry", "compliance", "news")


def route(question: str):
    """Pick a tool family from the question. A keyword list is a stand-in for lesson 5.5's
    trained router - and like that one, it is honest about being a stand-in."""
    external = any(w in question.lower() for w in EXTERNAL)
    return ("web", web_agent) if external else ("citations", guarded)


for q in ["What are the 2026 GDPR enforcement trends?",
          "Find all legal documents and estimate bulk processing cost"]:
    lane, _agent = route(q)
    print(f"  {lane:9} <- {q[:60]}")

In [ ]:
# WHICH ONE, AND WHY IT IS NOT A PREFERENCE
#
# Built-in grounding gives you citations Google stands behind and no scraping to maintain. It
# also bills per SEARCH QUERY on Gemini 3, not per prompt (lesson 6.3), so one question that
# makes the model search four times costs four.
#
# It also has a limit that decides the design. Binding google_search AND your own tools in one
# request is documented on the Gemini Developer API as Preview, Gemini 3 only - and the Vertex AI
# docs still say the combination is unsupported outright (checked 2026-09-04; lesson 6.3 quotes
# both pages). This notebook runs on Vertex. So do not plan on one request that searches the web
# and calls your functions: route ONE family per turn, which is what the agent below does anyway
# because the two paths answer different questions.
#
# Your own tool costs a provider relationship and a scraper's maintenance, and gives you the
# query text, a cache, a per-tenant rate limit, and something that works with no cloud at all.
#
# DocuMind uses both, in different places: grounding for the "what does the regulation say"
# path, where citations are the product, and the owned tool for the local lane, where there is
# no Google to ask. Neither is the answer; the routing is.

## Cell 9: Prove the rewrite changed nothing
Compare **tool calls**, not prose. The wording of a final answer differs between runs of the same model and tells you nothing; the sequence of tools chosen is the behaviour you are porting.

> When the lists differ, the cause is almost never the framework. It is a system prompt that got reworded during the move, or a tool description someone tidied. Diff those two first.

In [ ]:
# The gate for this lesson. A rewrite that changes behaviour is not a rewrite, it is a rewrite
# plus an undocumented bug. Run the same three questions through both implementations and
# compare the TOOL CALLS - not the prose, which will differ every time and does not matter.

QUESTIONS = [
    "Find all legal documents and estimate bulk processing cost",
    "How much would it cost to process 500 pages at standard rate?",
    "Show me this week's query stats",
]


def calls_from_langchain(agent, question):
    out = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return [tc["name"] for m in out["messages"] for tc in (getattr(m, "tool_calls", None) or [])]


print(f"{'question':52} {'6.4 (LangChain)':34}")
for q in QUESTIONS:
    got = calls_from_langchain(guarded, q)
    print(f"  {q[:50]:52} {str(got)[:34]}")

# Paste lesson 6.2's run_function_loop beside this and compare the two lists. Expect
# retrieve then calculate_processing_cost for the first question, calculate alone for
# the second, get_usage_stats for the third.
#
# When they differ - and on a first port they usually do - the cause is almost never the
# framework. It is that one of the two has a different system prompt, or a tool description
# that got reworded during the move. Diff those two things first.

## Cell 10: The production lane
This notebook is a teaching artefact. `deploy/services/chat/tools.py` is what ships, and it imports the tool functions from one module so lessons 6.1-6.4 and the deployed chat service cannot disagree about what `calculate_processing_cost` means.

In [ ]:
# deploy/services/chat/tools.py - the production lane.
#
# The notebook above is a teaching artefact. The service that ships imports the tool functions
# from one module, so lessons 6.1-6.4 and the deployed chat service cannot disagree about what
# calculate_processing_cost means.

from langchain_core.tools import tool

USD_INR = 85
RATES = {"standard": 0.05, "priority": 0.12, "bulk": 0.03}


@tool
def calculate_processing_cost(
        total_pages: int, num_documents: int = 1,
        processing_type: str = "standard") -> dict:
    """Estimate document processing cost in USD and INR.

    Args:
        total_pages: Total page count across all documents
        num_documents: How many documents those pages span
        processing_type: standard, priority, or bulk
    """
    rate = RATES.get(processing_type, RATES["standard"])
    cost = total_pages * rate
    return {"num_documents": num_documents,
            "total_pages": total_pages,
            "processing_type": processing_type,
            "rate_per_page": rate,
            "cost_usd": round(cost, 2),
            "cost_inr": round(cost * USD_INR, 2)}


# retrieve in this file calls the rag-api /v1/query endpoint from Module 7 rather than
# a mock dict. That is the only difference between the notebook's tools and production's, and
# it is deliberately the only one: everything else about the loop has already been proven here.
#
# Step 7's switch ships too (gap G3, 2026-09-05). deploy/shared/profile.py is this notebook's
# build_llm() / build_store(), and documind_tools.retrieve() reads the same DOCUMIND_PROFILE:
# `make chat-local` runs the deployed chat service against Chroma + Ollama gemma3:4b with no
# cloud credential - the Rs 0 lane 13.2 hands you. And the tenant reaches the production tool
# through the ToolRuntime the framework injects (its .context is what agent.py passes to
# agent.invoke(context=...)) - hidden from the model, and actually delivered.

## What to take to the next framework

| Before you adopt one | For this port |
|---|---|
| List your non-happy-path behaviour | Blocked operations, per-tool timeouts, audit logging |
| Find where each lives in the new world | All three moved into `wrap_tool_call` middleware |
| Find what the framework cannot do | No dispatch seam; guards must be re-expressed |
| Write the parity test before the port | Three questions, compared on tool calls |
| Name the capability you are buying | An offline lane, and one tool definition instead of two |

None of that is about LangChain. Run it against the next framework somebody proposes and you get a real answer in an afternoon instead of an opinion.